---
title: "AI and Archaeology"
subtitle: "Lecture Notes"
author: "Casey H. Shi"
date: today
format:
  html:
    theme: cosmo
    toc: true
    toc-depth: 2
    toc-location: left
    number-sections: true
    number-depth: 1
    code-copy: true
    code-overflow: wrap
    fig-align: center
jupyter: python3
execute:
  warning: false
---

In [ ]:
#| include: false
import collections, random, re, textwrap, urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image, ImageDraw, ImageOps

plt.rcParams["figure.dpi"] = 110
random.seed(42)
np.random.seed(42)
for folder in ("images", "data", "models"):
    Path(folder).mkdir(exist_ok=True)

# Image placeholders: any images/*.png named in this file that does not exist yet
# gets a grey stand-in. Save a real image under the same name and it takes over.
for qmd in Path(".").glob("*.qmd"):
    for caption, path in re.findall(r"!\[(.*?)\]\((images/[^)]+)\)", qmd.read_text(encoding="utf-8")):
        if not Path(path).exists():
            fig = plt.figure(figsize=(9.6, 4.4), facecolor="#eceff1")
            label = f"{Path(path).name}\n\n{textwrap.fill(caption, 60)}"
            fig.text(0.5, 0.5, label, ha="center", va="center", fontsize=14, color="#455a64", parse_math=False)
            fig.savefig(path, dpi=100)
            plt.close(fig)

::: {.callout-note collapse="true" title="How to run this page"}
**Install once:**

`pip install jupyter numpy pandas matplotlib seaborn scipy scikit-learn scikit-image pillow ultralytics segmentation-models-pytorch`

**Render:** `quarto preview lecture.qmd`

**Live demo:** run cells one at a time in VS Code or Positron, or make a notebook with `quarto convert lecture.qmd`.

Cells marked `#| eval: false` need files in `data/` and `models/`. They still run when you execute them by hand. Once the files are in place (Appendix A), replace `#| eval: false` with `#| eval: true` throughout so their outputs appear on the rendered page.
:::

## Lecture plan {.unnumbered}

| Min   | Section                                  | Activity                |
|-------|------------------------------------------|-------------------------|
| 0–4   | Why machine learning in archaeology?     | Discussion              |
| 4–12  | How machines learn                       | Demo                    |
| 12–22 | One dataset, two questions               | Demos, Check-ins 1–2    |
| 22–29 | From neurons to CNNs                     | Demos, Check-in 3       |
| 29–38 | Tuning, overfitting, and evaluation      | Demo, Check-ins 4–5     |
| 38–49 | Case study 1: classification             | Demos, Check-ins 6–7    |
| 49–63 | Case study 2: object detection           | Demo, Check-in 8        |
| 63–73 | Case study 3: segmentation               | Demos, Check-in 9       |
| 73–78 | Transformers                             | Demos                   |
| 78–83 | Generative models and LLMs               | Demo, Check-in 10       |
| 83–90 | Ethics, perspectivism, fairness, wrap-up | Discussion, Check-in 11 |

## Reading map {.unnumbered}

| Reading | Where | What to look for |
|----|----|----|
| Sakai et al. 2024 | @sec-why, @sec-cnn | Small training sets, probability maps, expert screening |
| Trier et al. 2019 | @sec-tuning, @sec-case2 | Transfer learning, confusion classes, why validation accuracy can mislead |
| Buławka et al. 2024 | @sec-case2 | YOLOv9, synthetic data, filtering false positives |
| Shi & Haag 2026 | @sec-case2 | Moving from detections to anthropological argument |
| Casini et al. 2023 | @sec-case3 | Segmentation, thresholds, human–AI workflow |
| Assael et al. 2022 | @sec-transformers | Transformers, restoration, historians working with a model |

# Why machine learning in archaeology? {#sec-why}

*≈ 4 min*

It took almost a century to record 430 figurative Nazca geoglyphs. One AI-assisted campaign added 303 more within six months of fieldwork (Sakai et al. 2024).

![A relief geoglyph of a killer whale holding a knife, one of the newly recorded Nazca figures, from Sakai et al. 2024](images/01-nazca-orca.png){width="75%"}

What happened is a funnel, and the funnel is the lesson:

1.  A neural network pre-trained on everyday photographs was fine-tuned on a few hundred known geoglyphs.
2.  Across about 629 km² of aerial imagery, it proposed 47,410 candidate patches.
3.  Experts screened these to 1,309 high-potential candidates, about 1 in 36.
4.  Fieldwork recorded 303 new figurative geoglyphs: 178 suggested individually by the model, 66 inside groups it flagged, and 59 found without it. Another 968 candidates were not reached in that season.

![Funnel diagram: 629 square kilometres, 47,410 AI candidates, 1,309 screened candidates, 303 new geoglyphs](images/02-nazca-funnel.png){width="70%"}

The model made the search tractable. People screened, walked, and confirmed. Three questions run through today's lecture:

- **What did the model learn from?** Labels are made by people, with their own assumptions.
- **How do we know it works?** The metric has to match the archaeological question.
- **What happens to the output?** Maps of sites have consequences.

# How machines learn {#sec-learn}

*≈ 8 min*

In ordinary programming we write the rules. In machine learning we supply examples and the computer estimates the rules.

::: {layout-ncol="2"}
![Nested circles: artificial intelligence contains machine learning, which contains deep learning, each with an archaeological example](images/03-ai-ml-dl.png)

![Hand-written rules compared with rules learned from labelled examples, using template matching and a CNN](images/04-rules-vs-learning.png)
:::

Trier et al. (2019) describe this shift. Template matching worked for pitfall traps that stood out clearly, but struggled when monuments varied in form. For Norwegian charcoal-burning platforms, it found 70% with a 72% false-positive rate. A pre-trained CNN with a support vector machine found 86% at 37%.

## Three ingredients

1.  A **model** with adjustable parameters: $\hat{y} = wx + b$
2.  A **loss** that measures error: $L = \frac{1}{n}\sum_{i=1}^{n}(\hat{y}_i - y_i)^2$
3.  An **optimiser** that lowers the loss: $w \leftarrow w - \eta \frac{\partial L}{\partial w}$

Here $\eta$ is the learning rate, the size of each downhill step.

![Gradient descent shown as steps down a loss surface, with small and large learning rates](images/05-gradient-descent.png){width="65%"}

**Demo: load the data.** We use 91 Archaic dart points from Fort Hood, Texas, from the R package `archdata`. `B.Width` has 22 missing values, so we leave it out.

In [ ]:
URL = "https://vincentarelbundock.github.io/Rdatasets/csv/archdata/DartPoints.csv"
csv = Path("data/DartPoints.csv")
if not csv.exists():
    urllib.request.urlretrieve(URL, csv)

FEATURES = ["Length", "Width", "Thickness", "J.Width", "H.Length", "Weight"]
points = pd.read_csv(csv).dropna(subset=FEATURES).reset_index(drop=True)
points[["Name"] + FEATURES].head(3)

**Demo: gradient descent.** Predict weight from length with the three ingredients and nothing else.

In [ ]:
x = ((points["Length"] - points["Length"].mean()) / points["Length"].std()).to_numpy()
y_weight = points["Weight"].to_numpy()

w, b, lr = 0.0, 0.0, 0.1
losses = []
for step in range(40):
    error = (w * x + b) - y_weight          # how wrong are we?
    losses.append((error ** 2).mean())      # loss: mean squared error
    w -= lr * 2 * (error * x).mean()        # step downhill in w
    b -= lr * 2 * error.mean()              # step downhill in b

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3.2))
ax1.plot(losses)
ax1.set(xlabel="step", ylabel="loss (MSE)", title="Learning: the loss goes down")
ax2.scatter(x, y_weight, s=15, alpha=0.6)
ax2.plot(np.sort(x), w * np.sort(x) + b, c="crimson")
ax2.set(xlabel="length (standardised)", ylabel="weight (g)", title=f"w = {w:.2f}, b = {b:.2f}")
plt.tight_layout()

*Try it:* set `lr = 0.001`, then `lr = 1.1`. The first barely moves. The second diverges. The learning rate is our first **hyperparameter**.

## Statistical learning in one idea

We care about error on **new** data, not the data we already have. For squared error:

$$\text{expected error} = \text{bias}^2 + \text{variance} + \text{irreducible noise}$$

- **Bias:** the model is too simple.
- **Variance:** the model is too sensitive to this particular sample.
- **Noise:** in archaeology this includes taphonomy, survey conditions, and disagreement between specialists.

## Machine learning or deep learning?

|   | Classical machine learning | Deep learning |
|----|----|----|
| Input | Measurements or hand-built features | Raw pixels, text, point clouds |
| Features | Chosen by the analyst | Learned in layers |
| Typical data | Tens to thousands of rows | Thousands of examples, or fewer with transfer learning |
| Example | Random forest on dart-point measurements | CNN on satellite chips |
| Main risk | Missing the informative variables | Overfitting, hard to inspect |

# One dataset, two questions {#sec-tabular}

*≈ 10 min*

![Fort Hood dart point types with the measured dimensions marked: length, width, thickness, juncture width, haft length](images/06-dart-points.png){width="70%"}

|   | Supervised | Unsupervised |
|----|----|----|
| Input | Examples **with** labels | Examples **without** labels |
| Question | Which type is this? | What structure is here? |
| Output | Class, box, mask, number | Clusters, low-dimensional map |
| Archaeology | Sherd to ware, chip to site | Morphological or compositional groups |
| Failure mode | Learns the labeller's bias | Finds patterns with no archaeological meaning |

![Side-by-side scatterplots: coloured labelled points with a boundary, and unlabelled points grouped into clusters](images/07-supervised-unsupervised.png){width="70%"}

## Supervised: random forest

A decision tree asks a sequence of yes-or-no questions. A random forest trains many trees on resampled data and lets them vote. This is **ensemble learning**.

![One decision tree splitting on haft length and width, then many trees voting as a random forest](images/08-random-forest.png){width="70%"}

**Demo: classify dart points from their measurements.**

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

X, y = points[FEATURES], points["Name"]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
rf = RandomForestClassifier(n_estimators=300, random_state=0)

scores = cross_val_score(rf, X, y, cv=cv)
print(f"Random forest, 5-fold accuracy: {scores.mean():.2f} ± {scores.std():.2f}")
print(f"Baseline, always guess the commonest type: {y.value_counts(normalize=True).max():.2f}")

rf.fit(X, y)
pd.Series(rf.feature_importances_, index=FEATURES).sort_values().plot.barh(
    figsize=(5, 2.6), title="Which measurements matter?")
plt.tight_layout()

Always read a score against a baseline. Guessing Pedernales every time already scores 34%.

## Unsupervised: PCA and k-means

**PCA** rotates the axes towards the directions of greatest variance, so we can plot many variables in two dimensions. **k-means** alternates between assigning points to the nearest centre and moving each centre.

::: {layout-ncol="2"}
![PCA as a rotation of axes along the longest direction of a point cloud](images/09-pca.png)

![Three k-means steps: random centres, assignment, updated centres](images/10-kmeans.png)
:::

**Demo: the same points, without the labels.**

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

X_std = StandardScaler().fit_transform(X)     # PCA and k-means need comparable scales
pca = PCA(n_components=2)
pcs = pca.fit_transform(X_std)
clusters = KMeans(n_clusters=5, n_init=10, random_state=0).fit_predict(X_std)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharex=True, sharey=True)
sns.scatterplot(x=pcs[:, 0], y=pcs[:, 1], hue=y, s=35, ax=axes[0]).set_title("Typological labels")
sns.scatterplot(x=pcs[:, 0], y=pcs[:, 1], hue=clusters.astype(str), s=35, palette="Set2",
                ax=axes[1]).set_title("k-means, k = 5")
axes[0].set(xlabel=f"PC1 ({pca.explained_variance_ratio_[0]:.0%})",
            ylabel=f"PC2 ({pca.explained_variance_ratio_[1]:.0%})")
plt.tight_layout()

print(f"Agreement with typology (adjusted Rand index): {adjusted_rand_score(y, clusters):.2f}")
print("Silhouette by k:", {k: round(silhouette_score(
    X_std, KMeans(k, n_init=10, random_state=0).fit_predict(X_std)), 2) for k in range(2, 8)})

This is an old argument in a new form. In the Ford–Spaulding debate of the 1950s, Spaulding argued that statistics could discover artefact types, while Ford treated types as tools built for a purpose.

**Other methods to know:** hierarchical clustering, DBSCAN for dense spatial groups, and t-SNE or UMAP for visual maps. Autoencoders learn compressed representations; Parisotto et al. used variational autoencoders to cluster Roman pottery profiles.

![Latent-space map of Roman pottery profiles, from the ROCOPOT work of Parisotto et al.](images/11-rocopot-latent.png){width="65%"}

::: {.callout-note icon="false" title="Check-in 1: choose a first step"}
You have 20,000 LiDAR tiles from an unsurveyed valley and no site inventory. You want to see what kinds of terrain pattern exist before deciding what to map. What do you try first?

A.  Train a CNN classifier on the tiles
B.  Extract features with a pre-trained network, cluster them, and review the groups with a specialist
C.  Report test-set accuracy
D.  Train YOLO with default settings
:::

::: {.callout-tip collapse="true" title="Answer"}
**B.** A classifier and a detector both need labels. Clustering gives you groups to inspect, and those groups can inform a labelling scheme.
:::

::: {.callout-note icon="false" title="Check-in 2: clusters and types"}
k-means with k = 5 does not reproduce the five named dart-point types. What is the most defensible conclusion?

A.  The typology is wrong
B.  The clusters are the real prehistoric types
C.  The two partitions answer different questions, so compare them before judging either
D.  k-means failed and the result should be discarded
:::

::: {.callout-tip collapse="true" title="Answer"}
**C.** Our k-means saw only six size measurements. The dataset also records blade, base, shoulder, and haft form, which we did not use. A mismatch is a prompt for archaeological reasoning.
:::

# From neurons to CNNs {#sec-cnn}

*≈ 7 min*

A neuron is a weighted sum passed through a non-linear function:

$$\hat{y} = \sigma\left(\sum_j w_j x_j + b\right)$$

Stack neurons in layers and you have an **artificial neural network (ANN)**. For classification, the last layer turns scores into class probabilities with softmax, and training minimises cross-entropy: $L = -\log \hat{p}_{\text{true class}}$.

![A single neuron with inputs, weights, sum, and activation, beside a small network with one hidden layer](images/12-neuron-network.png){width="70%"}

**Demo: a small ANN on the same table.** Does it beat the forest on 90 rows?

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline

mlp = make_pipeline(StandardScaler(),
                    MLPClassifier(hidden_layer_sizes=(32,), max_iter=3000, random_state=0))
scores = cross_val_score(mlp, X, y, cv=cv)
print(f"Neural network, 5-fold accuracy: {scores.mean():.2f} ± {scores.std():.2f}")

## Why images need something else

A dense layer connects every pixel to every neuron, and it has no idea that a mound shifted ten pixels is still a mound. CNNs rest on three ideas:

1.  **Local filters:** a small kernel looks at one neighbourhood.
2.  **Shared weights:** the same kernel slides across the image.
3.  **Downsampling:** each layer sees a wider area, moving from edges to textures to shapes.

::: {layout-ncol="2"}
![Dense connections compared with a local convolutional filter sliding over an image](images/13-dense-vs-conv.png)

![Feature hierarchy from edges to textures to parts to a whole object, ideally using a mound or coin example](images/14-feature-hierarchy.png)
:::

**Demo: convolution on Greek coins from Pompeii.** This image ships with scikit-image.

In [ ]:
from scipy.signal import convolve2d
from skimage import data

coins = data.coins()
kernels = {
    "vertical edges": np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]),
    "horizontal edges": np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]]),
    "blur": np.ones((7, 7)) / 49,
}

fig, axes = plt.subplots(1, 4, figsize=(13, 3))
axes[0].imshow(coins, cmap="gray")
axes[0].set_title("input")
for ax, (name, kernel) in zip(axes[1:], kernels.items()):
    ax.imshow(convolve2d(coins, kernel, mode="same", boundary="symm"), cmap="gray")
    ax.set_title(name)
for ax in axes:
    ax.axis("off")
plt.tight_layout()

We chose these kernels by hand. A CNN learns the numbers in its kernels from training data.

::: {.callout-note icon="false" title="Check-in 3: dense or convolutional?"}
A 640 × 640 greyscale chip feeds a dense layer of 1,000 neurons. Roughly how many weights does that layer need?

A.  64,000
B.  1.3 million
C.  410 million
D.  410 billion
:::

::: {.callout-tip collapse="true" title="Answer"}
**C.** 640 × 640 = 409,600 inputs, multiplied by 1,000 neurons, gives about 410 million weights. A convolutional layer with 32 filters of size 3 × 3 on the same chip needs 320 parameters.
:::

## One backbone, three heads

The feature-extracting backbone can stay much the same. The last part of the network changes with the question.

![One CNN backbone feeding three heads: image label, bounding boxes, and pixel mask, shown on the same satellite chip](images/15-three-heads.png){width="80%"}

| Task | Question | Output | Typical models | Readings |
|----|----|----|----|----|
| Classification | What is this? | One label per image or patch | ResNet, EfficientNet | Trier, Sakai |
| Detection | Where are the objects? | Boxes with confidence | Faster R-CNN, YOLO | Buławka, Shi & Haag |
| Segmentation | Which pixels? | Mask | U-Net, MA-Net | Casini |

## Transfer learning

Archaeological training sets are small, so we start from a network trained on another task, replace the final layers, and fine-tune. Trier et al., Sakai et al., and Casini et al. all do this. It helps a great deal, but it does not replace good data: ImageNet contains minivans and balloons, not cut marks (Courtenay et al. 2024).

![Transfer learning: freeze or fine-tune a pre-trained backbone and replace the final layer with archaeological classes](images/16-transfer-learning.png){width="70%"}

# Tuning, overfitting, and evaluation {#sec-tuning}

*≈ 9 min*

**Parameters** are learned from data. **Hyperparameters** are choices we make before or after training.

| Model | Hyperparameters that matter most |
|----|----|
| Random forest | Number of trees, tree depth |
| CNN | Learning rate, batch size, epochs, image size, weight decay, augmentation |
| Detector | All of the above, plus chip size, overlap, and the confidence threshold |
| Segmenter | All of the above, plus the loss and the probability threshold |

**Demo: watch a model overfit.**

In [ ]:
from sklearn.model_selection import validation_curve
from sklearn.tree import DecisionTreeClassifier

depths = list(range(1, 13))
train_scores, val_scores = validation_curve(
    DecisionTreeClassifier(random_state=0), X, y,
    param_name="max_depth", param_range=depths, cv=cv)

plt.figure(figsize=(5.5, 3.2))
plt.plot(depths, train_scores.mean(axis=1), "o-", label="training")
plt.plot(depths, val_scores.mean(axis=1), "o-", label="cross-validation")
plt.xlabel("tree depth (model complexity)")
plt.ylabel("accuracy")
plt.legend()
plt.tight_layout()

Training accuracy climbs towards 1. The question is what the cross-validation curve does. A gap between the two curves means the model is memorising.

![Learning curves showing a good fit, underfitting, and overfitting, for example Figure 1 of Courtenay et al. 2024](images/17-learning-curves.png){width="75%"}

**Ways to reduce overfitting:** more or better data, transfer learning, augmentation, early stopping, weight decay, dropout, or a simpler model.

## Three splits, and why geography matters

- **Training set:** fits the weights.
- **Validation set:** chooses hyperparameters and the best epoch.
- **Test set:** used once, for the final report.

Neighbouring chips look alike. If they fall on both sides of a split, the test score reflects memory of a place. Split by area, site, or scene, and if the model is meant to travel, test it somewhere new.

![Random chip split compared with a spatial block split on a map of one survey region](images/18-spatial-split.png){width="75%"}

::: {.callout-note icon="false" title="Check-in 4: read the curve"}
Training loss keeps falling, while validation loss bottoms out at epoch 40 and then climbs. You also want to compare ten configurations. What should you do?

A.  Compare the configurations on the test set, since it is the fairest judge
B.  Keep the epoch-40 weights, compare configurations on the validation set, and report the test set once at the end
C.  Train longer, because validation loss will recover
D.  Merge validation and test data to gain examples
:::

::: {.callout-tip collapse="true" title="Answer"}
**B.** Using one set for both selection and final scoring inflates results. Courtenay et al. (2024) raise this problem in published bone-surface-modification studies.
:::

## Metrics

| Metric | Formula | Archaeological meaning |
|----|----|----|
| Precision | TP / (TP + FP) | Of what the model flagged, how much is real? Low precision wastes field days. |
| Recall | TP / (TP + FN) | Of what is real, how much was found? Low recall misses heritage. |
| F1 | 2PR / (P + R) | Balance of the two |
| IoU | Overlap / union | How well do a predicted shape and a reference shape match? |
| mAP50 | Mean area under the precision–recall curve, with a match at IoU ≥ 0.5 | Standard summary for detectors |

![A confusion matrix beside a diagram of intersection over union with poor, fair, and good overlaps](images/19-confusion-iou.png){width="75%"}

## The accuracy trap

Trier et al. (2019) report a best validation accuracy of 99.46% for roundhouses. Their tables list 26 roundhouses and 1,842 background patches for validation. If that was the validation set, a model answering "background" every time would score 98.6%.

Mapping results tell a harder story. On Machrie Moor, one of their better areas, the model made 11 correct and 13 false roundhouse identifications against 15 known roundhouses. Those 15 were part of the learning set.

::: {.callout-note icon="false" title="Check-in 5: precision and recall"}
Using the Machrie Moor counts above, what are the precision and recall?

A.  Precision 73%, recall 46%
B.  Precision 46%, recall 73%
C.  Precision 46%, recall 87%
D.  Precision 87%, recall 73%
:::

::: {.callout-tip collapse="true" title="Answer"}
**B.** Precision = 11 / 24 = 46%. Recall = 11 / 15 = 73%. Trier et al. also report 87% false identifications, but that figure is 13 / 15, a rate relative to known monuments, so it is not 1 minus precision.
:::

# Case study 1: classifying artefacts and animal remains {#sec-case1}

*≈ 11 min*

Classification fits when the object is already isolated: one sherd, one coin, one mark under a microscope.

| Study | Material | Approach | Reported result |
|----|----|----|----|
| Pawlowicz & Downum 2021 | Over 3,000 Tusayan White Ware sherd images | CNN with rotation and scale augmentation | Comparable to, and sometimes above, four experts |
| Domínguez-Rodrigo et al. 2020 | Cut, tooth, and trampling marks | Transfer learning, CNN ensemble | 92%, disputed below |
| Zhang et al. 2024 (LUWA) | 23,130 microscope images of stone-tool wear | Pre-trained vision models including DINOv2 | Above human experts, with much room left |
| Wang & Deng 2024 | 30,222 oracle-bone characters | Benchmark from classic models to CNN | Random forest 64.9%, CNN 93.8% |

::: {layout-ncol="2"}
![Microscope images of a cut mark, a carnivore tooth mark, and a trampling mark](images/20-bsm-marks.png)

![Tusayan White Ware sherds arranged by type, from Pawlowicz and Downum 2021](images/21-tusayan-sherds.png)
:::

## Ensembles

- **Bagging:** many models on resampled data, as in a random forest.
- **Soft voting:** average the class probabilities of several networks.
- **Stacking:** a second-level model learns from first-level predictions. It needs its own data split.

![Diagram comparing bagging, soft voting, and stacking](images/22-ensembles.png){width="75%"}

## Augmentation

Augmentation creates plausible variations of training images. Trier et al. used flips and 90° rotations, which give eight versions of each terrain patch. It only adds variation you already expect, so each choice is an archaeological claim about what can vary.

**Demo: which of these could really occur?**

In [ ]:
crop = Image.fromarray(coins[10:110, 10:110]).resize((200, 200))
variants = {
    "original": crop,
    "mirror": ImageOps.mirror(crop),
    "rotate 90°": crop.rotate(90),
    "rotate 25°": crop.rotate(25),
    "darker": crop.point(lambda v: int(v * 0.6)),
    "zoomed crop": crop.crop((30, 30, 170, 170)).resize((200, 200)),
}
fig, axes = plt.subplots(1, 6, figsize=(13, 2.5))
for ax, (name, image) in zip(axes, variants.items()):
    ax.imshow(image, cmap="gray", vmin=0, vmax=255)
    ax.set_title(name, fontsize=10)
    ax.axis("off")
plt.tight_layout()

::: {.callout-note icon="false" title="Check-in 6: choose augmentations"}
You are training on Gupta archer-type coins. Which augmentation should you leave out?

A.  Small rotations
B.  Small brightness changes
C.  Mirror flips
D.  Slight crops
:::

::: {.callout-tip collapse="true" title="Answer"}
**C.** Mirroring reverses the legend and the figure, so the model trains on objects that do not exist. Flips are often reasonable for overhead imagery, though they do change the shadow direction.
:::

## Live demo: small published datasets

|   | Bone surface marks | Gupta coins | Oracle-MNIST (backup) |
|----|----|----|----|
| Publication | Domínguez-Rodrigo et al. 2020 | *Data in Brief* 57, 110934 (2024) | Wang & Deng 2024 |
| Task | Cut, tooth, or trampling mark? | Which ruler issued this coin? | Which of ten characters? |
| Size | 655 images (489 / 103 / 63) | 507 images, 18 classes (237 down to 2) | 30,222 images at 28 × 28 |
| Access | [Harvard Dataverse](https://doi.org/10.7910/DVN/62BRBP) | [GitHub](https://github.com/Dataset-of-Ancient-Coins/Dataset-of-Gupta-Archer-Type-Coins-with-Annotations) | [GitHub](https://github.com/wm-bupt/oracle-mnist) |

Both of the main datasets are imbalanced, which is normal in archaeology. Models were trained before class (Appendix B). The training call is short:

``` python
from ultralytics import YOLO
model = YOLO("yolo26n-cls.pt")          # pre-trained weights
model.train(data="data/bsm", epochs=60, imgsz=224, patience=20, fliplr=0.5, flipud=0.5)
```

In [ ]:
#| code-fold: true
#| code-summary: "Helper functions for the image demos"
IMG_EXT = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}

def images_in(folder):
    """All image files below a folder, in a stable order."""
    return sorted(p for p in Path(folder).rglob("*") if p.suffix.lower() in IMG_EXT)

def read_boxes(img):
    """YOLO labels for an image: rows of class, x-centre, y-centre, width, height (all 0 to 1)."""
    parts = ["labels" if part == "images" else part for part in img.parts]
    label = Path(*parts).with_suffix(".txt")
    if label.exists() and label.stat().st_size:
        return np.loadtxt(label, ndmin=2).reshape(-1, 5)
    return np.zeros((0, 5))

def draw_boxes(ax, img, boxes, color):
    """Show an image with YOLO-format boxes on top."""
    im = Image.open(img).convert("RGB")
    W, H = im.size
    ax.imshow(im)
    ax.axis("off")
    for _, cx, cy, w, h in boxes:
        ax.add_patch(plt.Rectangle(((cx - w / 2) * W, (cy - h / 2) * H), w * W, h * H,
                                   fill=False, ec=color, lw=1.5))

def box_masks(img, size):
    """Turn YOLO boxes into a filled-ellipse mask. Ellipses suit round mounds better than rectangles."""
    W, H = size
    mask = Image.new("L", size, 0)
    draw = ImageDraw.Draw(mask)
    for _, cx, cy, w, h in read_boxes(img):
        draw.ellipse([(cx - w / 2) * W, (cy - h / 2) * H, (cx + w / 2) * W, (cy + h / 2) * H], fill=1)
    return np.array(mask)

**Step 1: look at the training data.** Change `DATA` to switch datasets.

In [ ]:
#| eval: false
DATA = Path("data/bsm")          # data/bsm/train/<class>/, val/<class>/, test/<class>/
classes = sorted(p.name for p in (DATA / "train").iterdir() if p.is_dir())
print({c: len(images_in(DATA / "train" / c)) for c in classes})

fig, axes = plt.subplots(len(classes), 6, figsize=(11, 1.9 * len(classes)), squeeze=False)
for ax in axes.ravel():
    ax.axis("off")
for row, c in zip(axes, classes):
    files = images_in(DATA / "train" / c)
    for ax, path in zip(row, random.sample(files, min(6, len(files)))):
        ax.imshow(Image.open(path).convert("RGB"))
    row[0].set_title(c, loc="left", fontsize=11)
plt.tight_layout()

**Step 2: predict on random test images.** Re-run the cell for a new draw. Green is right and red is wrong.

In [ ]:
#| eval: false
from ultralytics import YOLO

model_a = YOLO("models/bsm_a.pt")
sample = random.sample(images_in(DATA / "test"), 8)

fig, axes = plt.subplots(2, 4, figsize=(12, 5.5))
for ax, path in zip(axes.ravel(), sample):
    result = model_a(str(path), verbose=False)[0]
    pred = result.names[result.probs.top1]
    conf = float(result.probs.top1conf)
    true = path.parent.name
    ax.imshow(Image.open(path).convert("RGB"))
    ax.axis("off")
    ax.set_title(f"{pred} ({conf:.0%})\ntrue: {true}", fontsize=10,
                 color="seagreen" if pred == true else "crimson")
plt.tight_layout()

**Step 3: soft voting, then look past accuracy.**

In [ ]:
#| eval: false
import torch
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

models = [model_a, YOLO("models/bsm_b.pt")]
names = model_a.names

y_true, y_pred = [], []
for path in images_in(DATA / "test"):
    probs = torch.stack([m(str(path), verbose=False)[0].probs.data for m in models]).mean(0)
    y_true.append(path.parent.name)
    y_pred.append(names[int(probs.argmax())])

print(f"Baseline, always the commonest class: {pd.Series(y_true).value_counts(normalize=True).max():.2f}")
print(classification_report(y_true, y_pred, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_true, y_pred, xticks_rotation=45, cmap="Blues")
plt.tight_layout()

## A live debate: how far can we trust 92%?

- **The claim:** Domínguez-Rodrigo et al. (2020) reported 92% on cut, tooth, and trampling marks.
- **The re-test:** Courtenay et al. (2024) re-ran three published datasets with separate validation and test sets. Most algorithms overfitted, and this dataset fell to 86%. Many images were poorly focused, cut marks most of all, so a model might be learning image quality. They also argued that stacking had been done without the extra data split it needs.
- **The rejoinder:** the original team replied in 2025 that these were pilot datasets, and that the re-test left too little data for training and validation.

Whichever side you take, the questions apply to any small dataset: what did the model see, and how was it tested?

![Grad-CAM maps where the network focuses away from the mark, for example Figure 5 of Courtenay et al. 2024](images/23-gradcam.png){width="70%"}

::: {.callout-note icon="false" title="Check-in 7: a real confusion matrix"}
A CNN is tested on 146 cut marks and 13 crocodile tooth marks. It predicts "cut mark" for all 159. What are its accuracy and its crocodile recall?

A.  8% and 100%
B.  92% and 0%
C.  92% and 92%
D.  50% and 50%
:::

::: {.callout-tip collapse="true" title="Answer"}
**B.** Accuracy = 146 / 159 = 91.8%, yet not one crocodile mark was found. This confusion matrix comes from Courtenay et al. (2024), Table 5.
:::

# Case study 2: object detection of archaeological sites {#sec-case2}

*≈ 14 min*

Detection answers *where are the features?* It returns boxes, classes, and confidence scores.

|   | Trier et al. 2019 | Buławka et al. 2024 | Shi & Haag 2026 |
|----|----|----|----|
| Imagery | ALS terrain model, Arran | HEXAGON, also tested on CORONA | HEXAGON and SkySat, Turpan |
| Method | ResNet18 patch classifier run across tiles | YOLOv9 with real, augmented, and synthetic data | YOLO-based detector |
| Target | Roundhouses, shieling huts, small cairns | Qanat shafts | Upcast mounds of *kariz* |
| Result | Encouraging on Machrie Moor, chaotic in Glen Shurig | Precision above 0.881, recall 0.627 in most tests | Precision 87.7%; over 160,000 features in ca. 500 km², 1974–2019 |

Trier et al. and Sakai et al. produce probability maps from patch classifiers, not boxes. So there are three routes to a map: classify patches, detect boxes, or segment pixels.

::: {layout-ncol="2"}
![Cross-section of a qanat or kariz with its line of vertical shafts and upcast mounds](images/24-qanat-section.png)

![The same area in HEXAGON and SkySat imagery, showing change in visible upcast mounds](images/25-hexagon-skysat.png)
:::

The payoff is interpretive. Shi & Haag use box shape, size, and distribution across four time slices to argue about changing management and collaboration among communities. A detection is evidence for an argument, not the end of the analysis.

![Maps of detected features across the four time slices, from Shi and Haag 2026](images/26-turpan-results.png){width="80%"}

## Two detector families

|   | Faster R-CNN (two-stage) | YOLO (one-stage) |
|----|----|----|
| Flow | Propose regions, then classify and refine each | Predict boxes and classes in one pass |
| Duplicates | Non-maximum suppression (NMS) | NMS in many versions; YOLO26 is NMS-free |
| Practical appeal | Long-established, flexible baseline | Fast and simple to iterate |
| Archaeology | Verschoof-van der Vaart & Lambers 2019 | Buławka et al. 2024, Shi & Haag 2026 |

::: {layout-ncol="2"}
![Faster R-CNN stages: backbone, region proposals, classification and box refinement](images/27-faster-rcnn.png)

![YOLO predicting boxes and classes directly from feature maps at several scales](images/28-yolo.png)
:::

## The pipeline

![Pipeline: imagery, georeferencing, annotation, tiling, spatial split, augmentation, training, tiled inference, filtering, GIS, field checks](images/29-detection-pipeline.png){width="90%"}

1.  **Prepare imagery.** Georeference scenes and keep the original resolution.
2.  **Annotate.** Label every instance in every chip, because unlabelled objects teach the model they are background.
3.  **Tile.** Chips of 640 pixels are a common choice. Overlap by at least the largest feature.
4.  **Split.** Keep areas separate. Ratios vary: 70/20/10 is a common start, while Casini et al. held out 10% and took 10% of the rest for validation.
5.  **Balance background.** Ultralytics suggests about 0–10% background images for detectors. Hard negatives matter more than the exact ratio: Casini et al. asked archaeologists to choose empty urban, agricultural, flooded, and rocky areas.
6.  **Augment.** Use flips, rotations, brightness, and mosaic. Hue and saturation mean nothing on panchromatic HEXAGON. Buławka et al. added synthetic images.
7.  **Train.** Start from pre-trained weights and default settings, then change one thing at a time.
8.  **Infer.** Tile the scene with overlap, predict, map boxes back to scene coordinates, and remove duplicates.
9.  **Filter and verify.** Buławka et al. labelled single shafts and pairs of shafts as separate classes, which removed many isolated false positives.
10. **Analyse.** Check samples in the field, then ask what the distribution means.

![Scene cut into overlapping chips, with one feature split by a chip edge and captured whole by the overlap](images/30-tiling-overlap.png){width="70%"}

In [ ]:
#| eval: false
def tile(scene, out_dir, size=640, overlap=160):
    """Cut a large scene into overlapping chips; the file name keeps each chip's offset."""
    Image.MAX_IMAGE_PIXELS = None                     # scanned frames can be huge
    im = Image.open(scene)
    step = size - overlap
    for top in range(0, im.height, step):
        for left in range(0, im.width, step):
            chip = im.crop((left, top, left + size, top + size))
            chip.save(Path(out_dir) / f"{Path(scene).stem}_{left}_{top}.png")

For GeoTIFFs, use `rasterio` windows so each chip keeps its coordinates. Training then takes a few lines, and each augmentation choice has a reason:

``` python
from ultralytics import YOLO
model = YOLO("yolo26n.pt")
model.train(data="data/chips/chips.yaml", imgsz=640, epochs=200, patience=30, batch=16,
            fliplr=0.5, flipud=0.5, degrees=90,   # overhead view: no fixed "up"
            hsv_h=0.0, hsv_s=0.0)                 # panchromatic: no hue or saturation
```

## Live demo: annotated chips and a trained YOLO model

**Step 1: what did the humans mark?**

In [ ]:
#| eval: false
from ultralytics import YOLO

CHIPS = Path("data/chips")                  # images/ and labels/ in YOLO format
detector = YOLO("models/sites_yolo.pt")     # your trained weights
chips = images_in(CHIPS / "images")

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, chip in zip(axes.ravel(), random.sample(chips, 8)):
    draw_boxes(ax, chip, read_boxes(chip), "lime")
fig.suptitle("Training chips with human annotations")
plt.tight_layout()

**Step 2: human and model, side by side.** Change `CONF` and run it again.

In [ ]:
#| eval: false
CONF = 0.25
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for col, chip in zip(axes.T, random.sample(chips, 4)):
    result = detector(str(chip), conf=CONF, verbose=False)[0]
    pred = np.c_[result.boxes.cls.cpu().numpy(), result.boxes.xywhn.cpu().numpy()]
    draw_boxes(col[0], chip, read_boxes(chip), "lime")
    col[0].set_title("human", fontsize=10)
    draw_boxes(col[1], chip, pred, "red")
    col[1].set_title(f"model: {len(pred)} at conf ≥ {CONF}", fontsize=10)
plt.tight_layout()

**Step 3: the threshold is an archaeological decision.** The curve shows how many features "exist" at each setting.

In [ ]:
#| eval: false
confs = np.concatenate([detector(str(c), conf=0.01, verbose=False)[0].boxes.conf.cpu().numpy()
                        for c in chips])
thresholds = np.linspace(0.05, 0.95, 19)

plt.figure(figsize=(5.5, 3.2))
plt.plot(thresholds, [(confs >= t).sum() for t in thresholds], "o-")
plt.xlabel("confidence threshold")
plt.ylabel("detections in these chips")
plt.tight_layout()

metrics = detector.val(data=str(CHIPS / "chips.yaml"), split="val", verbose=False)
print(f"precision {metrics.box.mp:.3f}  recall {metrics.box.mr:.3f}  mAP50 {metrics.box.map50:.3f}")

A desk-based inventory may accept a low threshold and more checking. A short field season may need a high one. Say which you chose and why.

## When detectors fail

On Arran, the same approach gave meaningful roundhouse results on Machrie Moor and chaotic results in Glen Shurig, where natural bumps resembled the targets. Trier et al. recommend adding false positives as **confusion classes**, as a fieldworker learns what merely looks like a monument.

![Machrie Moor and Glen Shurig results from Trier et al. 2019, side by side](images/31-arran-contrast.png){width="80%"}

::: {.callout-note icon="false" title="Check-in 8: chips"}
Your largest features are about 150 pixels across. You cut scenes into 640-pixel chips with no overlap, and some features are split at chip edges. What fixes this?

A.  Train for more epochs
B.  Overlap chips by at least 150 pixels, map boxes back to the scene, and remove duplicates
C.  Raise the confidence threshold
D.  Turn on mosaic augmentation
:::

::: {.callout-tip collapse="true" title="Answer"}
**B.** Overlap ensures every feature appears whole in at least one chip. Overlap also creates duplicates, so they must be merged after boxes return to scene coordinates.
:::

# Case study 3: segmentation and the site problem {#sec-case3}

*≈ 10 min*

Segmentation asks *which pixels belong to the site?* That immediately raises a harder question: **where does a site end?**

A Mesopotamian tell may have a low outer town or artefact scatter that extends its occupied area several times over (Wilkinson, quoted in Casini et al. 2023). Archaeologists have long questioned the site as a unit (Dunnell 1992; McCoy 2020). A box hides this. A probability map at least makes the decision visible.

![A tell with its low outer town and surface scatter, with several possible boundaries drawn on the same image](images/32-tell-boundary.png){width="75%"}

|   | Semantic segmentation | Instance segmentation |
|----|----|----|
| Output | Every pixel gets a class | Each object gets its own mask |
| Touching sites | Merge into one region | Stay separate |
| Good for | Extent, land cover, site area | Counting, measuring individual features |

## Which models are best here?

| Model | Output | Threshold you control | Best for |
|----|----|----|----|
| **U-Net with a pre-trained encoder** | Probability per pixel | Per pixel | **Default for site extents** |
| MA-Net, DeepLabV3+, SegFormer | Probability per pixel | Per pixel | Second step, once a baseline exists |
| Mask R-CNN, YOLO-seg | One mask per object | Per object | Counting discrete features |
| YOLO26-sem | Class ID per pixel | Not in its standard output | Fast dense maps at a fixed boundary |
| SAM family | Mask from a box or point prompt | Not a site model | Turning boxes into draft masks |

**Recommendation:** start with a U-Net and a pre-trained ResNet-34 encoder, using Dice plus binary cross-entropy loss. Keep the probability raster and threshold it late. In Maya LiDAR data, U-Net beat Mask R-CNN on both tasks tested (Bundzel et al. 2020). Casini et al. compared U-Net with MA-Net, two encoders, and two losses, and the differences were small. Data and labels will probably matter more than the architecture.

![U-Net architecture: encoder, decoder, and skip connections](images/33-unet.png){width="70%"}

**Demo: a threshold decides how many objects exist.** The scikit-image documentation notes that a simple threshold does not separate these coins cleanly, which suits our purpose.

In [ ]:
from skimage import filters, measure

otsu = filters.threshold_otsu(coins)
fig, axes = plt.subplots(1, 4, figsize=(13, 3.2))
axes[0].imshow(coins, cmap="gray")
axes[0].set_title("Pompeii coins")
for ax, t in zip(axes[1:], (otsu - 30, otsu, otsu + 30)):
    mask = coins > t
    n = sum(region.area >= 100 for region in measure.regionprops(measure.label(mask)))
    ax.imshow(mask, cmap="gray")
    ax.set_title(f"threshold {t}: {n} objects")
for ax in axes:
    ax.axis("off")
plt.tight_layout()

**Demo: from boxes to masks.** We can reuse Case study 2's labels as ellipses, or use boxes as prompts for SAM. Both give weak labels, so inspect them, especially on faint features.

In [ ]:
#| eval: false
from ultralytics import SAM

sam = SAM("mobile_sam.pt")          # small enough for a laptop
chip = random.choice([c for c in chips if len(read_boxes(c))])
im = Image.open(chip).convert("RGB")
W, H = im.size
boxes = read_boxes(chip)
xyxy = [[(cx - w / 2) * W, (cy - h / 2) * H, (cx + w / 2) * W, (cy + h / 2) * H]
        for _, cx, cy, w, h in boxes]
sam_mask = sam(str(chip), bboxes=xyxy, verbose=False)[0].masks.data.any(0).cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
draw_boxes(axes[0], chip, boxes, "lime")
axes[0].set_title("human boxes")
for ax, mask, title in zip(axes[1:], (box_masks(chip, (W, H)), sam_mask),
                           ("ellipses from boxes", "SAM, prompted by boxes")):
    ax.imshow(im)
    ax.imshow(np.ma.masked_equal(mask.astype(float), 0), cmap="autumn", alpha=0.5)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()

**Demo: a threshold you control.** The U-Net was trained on these masks before class (Appendix B). Casini et al. also blur predictions before thresholding.

In [ ]:
#| eval: false
import segmentation_models_pytorch as smp
import torch
from scipy.ndimage import gaussian_filter

SIZE = 512
MEAN = torch.tensor([0.485, 0.456, 0.406])[:, None, None]
STD = torch.tensor([0.229, 0.224, 0.225])[:, None, None]
device = "cuda" if torch.cuda.is_available() else "cpu"

unet = smp.Unet("resnet34", encoder_weights=None, in_channels=3, classes=1).to(device)
unet.load_state_dict(torch.load("models/sites_unet.pt", map_location=device))
unet.eval()

chip = random.choice(chips)
im = Image.open(chip).convert("RGB").resize((SIZE, SIZE))
x = (torch.from_numpy(np.array(im)).permute(2, 0, 1).float() / 255 - MEAN) / STD
with torch.no_grad():
    prob = unet(x[None].to(device)).sigmoid()[0, 0].cpu().numpy()
prob = gaussian_filter(prob, sigma=2)

fig, axes = plt.subplots(1, 5, figsize=(15, 3.2))
axes[0].imshow(im)
axes[0].set_title("chip")
axes[1].imshow(prob, vmin=0, vmax=1, cmap="magma")
axes[1].set_title("P(site) per pixel")
for ax, t in zip(axes[2:], (0.3, 0.5, 0.7)):
    ax.imshow(im)
    ax.contour((prob > t).astype(float), levels=[0.5], colors="cyan", linewidths=1)
    ax.set_title(f"t = {t}: {measure.label(prob > t).max()} regions")
for ax in axes:
    ax.axis("off")
plt.tight_layout()

Polygons can then go to a GIS with `rasterio.features.shapes` or GDAL.

## Reading Casini et al. 2023

- **Model:** MA-Net, a U-Net with attention, plus EfficientNet-B3 and focal loss, fine-tuned on almost 5,000 surveyed sites.
- **Scores:** IoU of 0.8154. Automatic detection accuracy was about 62%, rising to about 80% once experts accounted for sites no longer visible in modern imagery.
- **Operational test:** in Maysan the model recovered 17 of 20 sites and suggested about 20 more shapes, most easily dismissed by an expert.
- **Transfer test:** in Uzbekistan only about 25–30% of sites were correctly identified.
- **Lesson:** the output is a heatmap or a vector layer for an expert, and corrected results feed the next round of training.

::: {layout-ncol="2"}
![Probability overlay for the Maysan test area, Figure 4 of Casini et al. 2023](images/34-casini-heatmap.png)

![Human–AI workflow: dataset, model, overlay or vectors, expert review, refined dataset, Figure 5 of Casini et al. 2023](images/35-human-loop.png)
:::

::: {.callout-note icon="false" title="Check-in 9: IoU or detection?"}
In Casini et al., the model with the higher IoU did worse at detecting sites. How can that happen?

A.  IoU and detection measure the same thing, so this must be an error
B.  A cautious model predicts less area: its masks fit well when they appear, but it finds fewer sites
C.  Higher IoU always signals overfitting
D.  The test set must have been smaller
:::

::: {.callout-tip collapse="true" title="Answer"}
**B.** Fewer positive pixels shrink the union, which can raise IoU. For prospection, a rough overlap with a real site may matter more than a perfect outline.
:::

# Transformers {#sec-transformers}

*≈ 5 min*

A CNN builds up from local neighbourhoods. A transformer lets every part of the input look at every other part.

1.  Split the input into **tokens**: characters, words, or image patches.
2.  Turn each token into a vector and add its position.
3.  Use **attention**: each token scores all the others and takes a weighted average of them.
4.  Stack these blocks and add a head for the task.

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^{\top}}{\sqrt{d_k}}\right)V$$

![Attention weights linking a damaged word to distant, relevant words in an inscription](images/36-attention.png){width="70%"}

**Demo: attention is a weighted lookup.** The tokens come from the Roman funerary formula *Dis Manibus … vixit annis*. These two-number vectors are invented; real ones are learned.

In [ ]:
tokens = ["DIS", "MANIBVS", "VIXIT", "ANNIS", "[gap]"]
E = np.array([[1.0, 0.1], [0.9, 0.2], [0.1, 1.0], [0.2, 0.9], [0.2, 0.8]])
scores = 5 * E @ E.T                                              # similarity of every pair
A = np.exp(scores) / np.exp(scores).sum(axis=1, keepdims=True)    # softmax, row by row

plt.figure(figsize=(4.8, 3.8))
sns.heatmap(A, annot=True, fmt=".2f", cmap="Blues", cbar=False, xticklabels=tokens, yticklabels=tokens)
plt.title("Attention: each row looks at the columns")
plt.tight_layout()

|   | CNN | Transformer |
|----|----|----|
| Looks at | Neighbourhoods that widen with depth | All tokens at once |
| Built-in assumptions | Strong: nearby pixels matter | Fewer: more is learned from data |
| Small datasets | Often fine with transfer learning | Usually needs strong pre-training |

## Applications in archaeology

| Project | Input | Result |
|----|----|----|
| **Ithaca** (Assael et al. 2022) | 78,608 Greek inscriptions | 62% restoration alone; historians rose from 25% to 72% with it; 71% for place; dates within 30 years |
| Aeneas (Assael et al. 2025) | Over 176,000 Latin inscriptions, text and images | 72% province accuracy across 62 provinces; dates within 13 years; finds parallels |
| Vesuvius Challenge 2023 | CT scans of carbonised scrolls | TimeSformer gave the clearest ink images, cross-checked with two other architectures |
| Casini et al. 2023 | Satellite tiles | Attention inside a U-Net-style network |
| LUWA (Zhang et al. 2024) | Microscope images of stone tools | Pre-trained vision models including DINOv2 outperformed experts |

The Ithaca result is the one to remember: the best performance came from historians working with the model.

::: {layout-ncol="2"}
![Ithaca's restoration hypotheses, geographic map, and date distribution for one inscription](images/37-ithaca.png)

![Ink detection on a Herculaneum scroll segment from the Vesuvius Challenge](images/38-vesuvius.png)
:::

**Live demo:** open [predictingthepast.com](https://predictingthepast.com/) and try a short inscription.

# Generative models and LLMs {#sec-generative}

*≈ 5 min*

| Family | Idea | Archaeological example |
|----|----|----|
| VAE | Compress to a latent space, then rebuild | Clustering Roman pottery profiles (Parisotto et al.) |
| GAN | A generator competes with a discriminator | Synthetic images for rare classes |
| Diffusion | Learn to remove noise step by step | Image generators used for reconstructions |
| LLM | Predict the next token | Text search, extraction, drafting |

![Four small diagrams showing how a VAE, GAN, diffusion model, and LLM work](images/39-generative-family.png){width="80%"}

Synthetic data does not always mean generative AI. Buławka et al. used artificial synthetic data for qanats, and Arch-I-Scan built synthetic data from 3D vessel models.

## How an LLM works

1.  Text is split into tokens, and each token becomes a vector.
2.  A transformer reads the context.
3.  The model outputs a probability for every possible next token.
4.  Training adjusts its parameters so the real next token becomes more likely.
5.  Generating text means sampling a token, appending it, and repeating.

None of these steps checks a fact. Retrieval can feed real documents to the model, which helps, but someone still has to read the source.

![Prompt to tokens, transformer, next-token probabilities, and a sampled token, repeated in a loop](images/40-llm-steps.png){width="75%"}

**Demo: a language model in ten lines.** It learns which letter tends to follow which in the opening of Caesar's *Gallic War*.

In [ ]:
text = ("gallia est omnis divisa in partes tres quarum unam incolunt belgae "
        "aliam aquitani tertiam qui ipsorum lingua celtae nostra galli appellantur")

counts = collections.defaultdict(collections.Counter)
for current, following in zip(text, text[1:]):
    counts[current][following] += 1

letter, output = "g", "g"
for _ in range(90):
    letters, weights = zip(*counts[letter].items())
    letter = random.choices(letters, weights)[0]
    output += letter
print(output)

The output looks vaguely Latin and means nothing. An LLM has the same core task, with a far longer context and vastly more parameters.

## In archaeology

| Use | Example | Caution |
|----|----|----|
| Searching grey literature | AGNES finds material in Dutch report full text that metadata miss (van den Dikkenberg & Brandsen 2025) | Evaluate against known sites |
| Translation | Akkadian to English, BLEU4 of 36.52 and 37.47, using a CNN-based model (Gutherz et al. 2023) | Some outputs were fluent but wrong |
| Contextualising inscriptions | Aeneas retrieves parallels from a curated corpus | Treat results as starting points |
| Images and reconstruction | Illustrating competing hypotheses (Magnani & Clindaniel 2023) | Outputs can reproduce old science (below) |

Magnani & Clindaniel (2025) compared AI descriptions and images of Neanderthals with scholarship from 1923 to 2023. The text most resembled literature from the early 1960s, and the images resembled work from the late 1980s and early 1990s.

![AI-generated Neanderthal scene with anachronisms marked, from Magnani and Clindaniel 2025](images/41-neanderthal-ai.png){width="70%"}

::: {.callout-note icon="false" title="Check-in 10: a fluent citation"}
An LLM writes a clear summary of a site report and cites a paper you cannot find. What is the right next step?

A.  Trust it, because the writing is good
B.  Ask the LLM whether it is sure
C.  Treat the claim as unverified, go to the primary report, and if you keep this workflow, retrieve source text first
D.  Cite it as a personal communication
:::

::: {.callout-tip collapse="true" title="Answer"}
**C.** Fluency is not evidence. Asking the same model to confirm itself does not verify anything.
:::

# Ethics, data perspectivism, and fairness {#sec-ethics}

*≈ 6 min*

## Whose labels?

Most supervised models treat labels as ground truth, often after a majority vote. **Data perspectivism** argues for keeping disagreement instead of erasing it (Cabitza et al. 2023). That suits archaeology. Casini et al. found it very subjective to decide whether a mask counted as a prediction, and Trier et al. argue that classification should be explicit whether it happens in a network or in an archaeologist's head.

In practice, store each annotator's label. Three site outlines can be averaged into a soft mask, and the threshold from Case study 3 becomes a question of how many specialists must agree.

![Three annotators' different outlines of one mound, with their average as a soft mask](images/42-annotator-disagreement.png){width="70%"}

## Whose past is visible?

Training data reflect where people surveyed and what they valued. Scotland's national record is partial and unrepresentative (Trier et al. 2019), and models trained on such records can inherit its gaps. Critics argue that models trained on well-studied regions will favour monumental remains over earthen, seasonal, or sacred places (Lamar & McClure 2025).

**Fairness in archaeology** can be tested: report recall by region, period, and site type, not only overall.

![World map of training data for archaeological AI compared with where sites are likely to exist](images/43-survey-bias-map.png){width="75%"}

## Who sees the result, and where did the data come from?

- **Looting and sacred places:** high-resolution maps can expose sites and sensitive landscapes (Cohen et al. 2020; Davis et al. 2021).
- **Consent:** in the 2025 OpenAI to Z Challenge, entrants produced predicted Amazonian "hotspots" from open data. Lamar & McClure criticised the lack of community consultation and of standards for ethics or error reporting.
- **Data governance:** FAIR asks whether data can be found and reused. CARE adds collective benefit, authority to control, responsibility, and ethics.
- **Provenance:** HEXAGON and CORONA are Cold War intelligence archives repurposed for archaeology (Hammer et al. 2022).

![A declassified HEXAGON frame beside a modern satellite image of the same landscape](images/44-spy-satellite.png){width="65%"}

::: {.callout-note icon="false" title="Check-in 11: three specialists disagree"}
Three specialists disagree about whether a faint scatter is a site. What would a perspectivist pipeline do?

A.  Take the majority vote and delete the minority view
B.  Ask only the most senior specialist
C.  Keep all three judgements, use the disagreement in training or evaluation, and report it
D.  Remove the example
:::

::: {.callout-tip collapse="true" title="Answer"}
**C.** Disagreement is information, and ambiguous cases may be the most interesting archaeologically.
:::

::: {.callout-important title="Discussion"}
Your model flags 40 likely tombs in a region with active looting. Who should see the coordinates, at what resolution, and who decides?
:::

## Before you publish a model

1.  Say who labelled the data and how disagreement was handled.
2.  Split by area or site, and keep the test set untouched.
3.  Report precision, recall, and results by group, not accuracy alone.
4.  Share code, weights, and data where that is safe.
5.  Decide location resolution with heritage authorities and descendant communities before release.
6.  Label AI-generated text and images as such.

# Wrap-up {#sec-wrap}

| Your question | Start with | Report |
|----|----|----|
| What is this object? | Pre-trained CNN classifier | Confusion matrix, recall per class |
| Where are these features? | YOLO or Faster R-CNN on overlapping chips | Precision, recall, mAP, field checks |
| How far does this site extend? | U-Net with pre-trained encoder | IoU, detection rate, threshold used |
| What groups exist? | PCA and clustering | Stability, archaeological review |
| What does this text say? | Specialised model or retrieval with an LLM | Expert verification against sources |

**Five things to keep:**

1.  Compare every score with a baseline.
2.  Split your data as the world will test it.
3.  Thresholds are archaeological decisions.
4.  A model is only as good as its labels, and labels are interpretations.
5.  The most successful systems here keep experts in the loop.

# References {.unnumbered}

## Required readings {.unnumbered}

- Assael, Y., Sommerschield, T., Shillingford, B., Bordbar, M., Pavlopoulos, J., Chatzipanagiotou, M., Androutsopoulos, I., Prag, J., & De Freitas, N. (2022). Restoring and attributing ancient texts using deep neural networks. *Nature*, 603(7900), 280–283. <https://doi.org/10.1038/s41586-022-04448-z>
- Buławka, N., Orengo, H. A., & Berganzo-Besga, I. (2024). Deep learning-based detection of qanat underground water distribution systems using HEXAGON spy satellite imagery. *Journal of Archaeological Science*, 171, 106053. <https://doi.org/10.1016/j.jas.2024.106053>
- Casini, L., Marchetti, N., Montanucci, A., Orrù, V., & Roccetti, M. (2023). A human–AI collaboration workflow for archaeological sites detection. *Scientific Reports*, 13(1), 8699. <https://doi.org/10.1038/s41598-023-36015-5>
- Sakai, M., Sakurai, A., Lu, S., Olano, J., Albrecht, C. M., Hamann, H. F., & Freitag, M. (2024). AI-accelerated Nazca survey nearly doubles the number of known figurative geoglyphs and sheds light on their purpose. *Proceedings of the National Academy of Sciences*, 121(40), e2407652121. <https://doi.org/10.1073/pnas.2407652121>
- Shi, C. H., & Haag, C. E. (2026). Pixelated transformations: Tracing changing irrigation management through deep-learning-based object detection on HEXAGON and SkySat satellite images. *Journal of Archaeological Method and Theory*, 33, 48. <https://doi.org/10.1007/s10816-026-09787-z>
- Trier, Ø. D., Cowley, D. C., & Waldeland, A. U. (2019). Using deep neural networks on airborne laser scanning data: Results from a case study of semi-automatic mapping of archaeological topography on Arran, Scotland. *Archaeological Prospection*, 26(2), 165–175. <https://doi.org/10.1002/arp.1731>

## Also cited {.unnumbered}

- Assael, Y., Sommerschield, T., Cooley, A., et al. (2025). Contextualizing ancient texts with generative neural networks. *Nature*, 645, 141–147. <https://doi.org/10.1038/s41586-025-09292-5>
- Bundzel, M., Jaščur, M., Kováč, M., Lieskovský, T., Sinčák, P., & Tkáčik, T. (2020). Semantic segmentation of airborne LiDAR data in Maya archaeology. *Remote Sensing*, 12(22), 3685. <https://doi.org/10.3390/rs12223685>
- Cabitza, F., Campagner, A., & Basile, V. (2023). Toward a perspectivist turn in ground truthing for predictive computing. *Proceedings of the AAAI Conference on Artificial Intelligence*, 37(6), 6860–6868. <https://doi.org/10.1609/aaai.v37i6.25840>
- Carroll, S. R., et al. (2020). The CARE Principles for Indigenous Data Governance. *Data Science Journal*, 19(1), 43. <https://doi.org/10.5334/dsj-2020-043>
- Cohen, A., Klassen, S., & Evans, D. (2020). Ethics in archaeological lidar. *Journal of Computer Applications in Archaeology*, 3(1), 76–91. <https://doi.org/10.5334/jcaa.48>
- Courtenay, L. A., Vanderesse, N., Doyon, L., & Souron, A. (2024). Deep learning-based computer vision is not yet the answer to taphonomic equifinality in bone surface modifications. *Journal of Computer Applications in Archaeology*, 7(1), 388–411. <https://doi.org/10.5334/jcaa.145>
- Davis, D. S., Buffa, D., Rasolondrainy, T., et al. (2021). The aerial panopticon and the ethics of archaeological remote sensing in sacred cultural spaces. *Archaeological Prospection*.
- Domínguez-Rodrigo, M., Cifuentes-Alcobendas, G., Jiménez-García, B., Abellán, N., Pizarro-Monzo, M., Organista, E., & Baquedano, E. (2020). Artificial intelligence provides greater accuracy in the classification of modern and ancient bone surface modifications. *Scientific Reports*, 10, 18862. <https://doi.org/10.1038/s41598-020-75994-7>
- Dunnell, R. C. (1992). The notion of site. In J. Rossignol & L. Wandsnider (Eds.), *Space, Time, and Archaeological Landscapes* (pp. 21–41). Plenum Press.
- Ford, J. A. (1954). The type concept revisited. *American Anthropologist*, 56(1), 42–54.
- Gupta archer-type coin dataset: A novel dataset of Gupta archer type coins for machine learning-based classification. (2024). *Data in Brief*, 57, 110934. <https://doi.org/10.1016/j.dib.2024.110934>
- Gutherz, G., Gordin, S., Sáenz, L., Levy, O., & Berant, J. (2023). Translating Akkadian to English with neural machine translation. *PNAS Nexus*, 2(5), pgad096. <https://doi.org/10.1093/pnasnexus/pgad096>
- Hammer, E., FitzPatrick, M., & Ur, J. (2022). Succeeding CORONA: Declassified HEXAGON intelligence imagery for archaeological and historical research. *Antiquity*, 96(387), 679–695. <https://doi.org/10.15184/aqy.2022.22>
- Lamar, A. K., & McClure, S. B. (2025). Data sovereignty vs. digital prospecting. *Public Seminar*. <https://publicseminar.org/2025/10/ai-in-the-amazon/>
- Magnani, M., & Clindaniel, J. (2023). Artificial intelligence and archaeological illustration. *Advances in Archaeological Practice*, 11(4), 452–460.
- Magnani, M., & Clindaniel, J. (2025). Artificial intelligence and the interpretation of the past. *Advances in Archaeological Practice*. <https://doi.org/10.1017/aap.2025.10110>
- McCoy, M. D. (2020). The site problem: A critical review of the site concept in archaeology in the digital age. *Journal of Field Archaeology*, 45(sup1), S18–S26. <https://doi.org/10.1080/00934690.2020.1713283>
- Núñez Jareño, S. J., van Helden, D. P., Mirkes, E. M., Tyukin, I. Y., & Allison, P. M. (2021). Learning from scarce information: Using synthetic data to classify Roman fine ware pottery. *Entropy*, 23, 1140. <https://doi.org/10.3390/e23091140>
- Parisotto, S., Leone, N., Schönlieb, C.-B., & Launaro, A. Unsupervised clustering of Roman potsherds via variational autoencoders. arXiv:2006.03156.
- Pawlowicz, L. M., & Downum, C. E. (2021). Applications of deep learning to decorated ceramic typology and classification: A case study using Tusayan White Ware from Northeast Arizona. *Journal of Archaeological Science*, 130, 105375. <https://doi.org/10.1016/j.jas.2021.105375>
- Rejoinder to Courtenay et al.: Reassessing deep learning (and meta-learning) computer vision as an efficient method to determine taphonomic agency in bone surface modifications. (2025). *bioRxiv*. <https://doi.org/10.1101/2025.01.31.635872>
- Spaulding, A. C. (1953). Statistical techniques for the discovery of artifact types. *American Antiquity*, 18(4), 305–313.
- van den Dikkenberg, L., & Brandsen, A. (2025). Using text mining to search for Neolithic Vlaardingen Culture sites in the Rhine-Meuse-Scheldt delta. *Journal of Computer Applications in Archaeology*.
- Verschoof-van der Vaart, W. B., & Lambers, K. (2019). Learning to look at LiDAR: The use of R-CNN in the automated detection of archaeological objects in LiDAR data from the Netherlands. *Journal of Computer Applications in Archaeology*. <https://doi.org/10.5334/jcaa.32>
- Wang, M., & Deng, W. (2024). A dataset of oracle characters for benchmarking machine learning algorithms. *Scientific Data*, 11, 87. <https://doi.org/10.1038/s41597-024-02933-w>
- Wilkinson, M. D., et al. (2016). The FAIR Guiding Principles for scientific data management and stewardship. *Scientific Data*, 3, 160018. <https://doi.org/10.1038/sdata.2016.18>
- Zhang, J., et al. (2024). LUWA dataset: Learning lithic use-wear analysis on microscopic images. *Proceedings of CVPR*, 22563–22573.

# Appendix A: files for the live demos {.unnumbered}

``` text
lecture.qmd
images/                  # placeholders appear here; overwrite them with real images
data/
  DartPoints.csv         # downloaded automatically
  bsm_raw/               # one folder per class, sorted by you
  bsm/                   # train/, val/, test/, made by Appendix B
  chips/
    chips.yaml
    images/              # your annotated chips
    labels/              # YOLO .txt files with matching names
models/
  bsm_a.pt, bsm_b.pt     # from Appendix B
  sites_yolo.pt          # your trained YOLO weights
  sites_unet.pt          # from Appendix B
```

Example `chips.yaml`:

``` yaml
path: data/chips
train: images/train
val: images/val
names:
  0: feature
```

**Preparation:**

1.  Download the bone-mark images from Dataverse and sort them into `data/bsm_raw/cut`, `tooth`, and `trampling`.
2.  For coins, keep the larger classes only. The script below skips classes with fewer than 30 images, leaving Chandragupta II, Kumaragupta I, Skandagupta, and Samudragupta.
3.  Run Appendix B once.
4.  Add your chips and `sites_yolo.pt`.
5.  Replace `#| eval: false` with `#| eval: true` and render.

# Appendix B: pre-lecture training script {.unnumbered}

Save as `train_models.py`. It does not run when the page renders. With only a few chips, the U-Net shows the pipeline but will not give a reliable result, so train it on a larger annotated set if you can.

``` python
import gzip, random, shutil, urllib.request
from pathlib import Path

import numpy as np
import segmentation_models_pytorch as smp
import torch
from PIL import Image, ImageDraw, ImageOps
from torch.utils.data import DataLoader, Dataset
from ultralytics import YOLO

IMG_EXT = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}
Path("models").mkdir(exist_ok=True)


def split_folder(src, dst, val=0.15, test=0.15, min_images=30, seed=0):
    """Pad images to squares and copy them to dst/{train,val,test}/<class>/."""
    rng = random.Random(seed)
    for cls in sorted(p for p in Path(src).iterdir() if p.is_dir()):
        files = sorted(p for p in cls.iterdir() if p.suffix.lower() in IMG_EXT)
        if len(files) < min_images:
            continue
        rng.shuffle(files)
        n_val, n_test = round(len(files) * val), round(len(files) * test)
        splits = {"test": files[:n_test],
                  "val": files[n_test:n_test + n_val],
                  "train": files[n_test + n_val:]}
        for split, part in splits.items():
            out = Path(dst) / split / cls.name
            out.mkdir(parents=True, exist_ok=True)
            for f in part:
                square = ImageOps.pad(Image.open(f).convert("RGB"), (256, 256), color=0)
                square.save(out / f"{f.stem}.png")


def oracle_to_folders(out="data/oracle_raw", per_class=300):
    """Backup dataset: save a sample of Oracle-MNIST as one folder per class."""
    base = "https://raw.githubusercontent.com/wm-bupt/oracle-mnist/main/data/oracle/"
    get = lambda name: gzip.decompress(urllib.request.urlopen(base + name).read())
    images = np.frombuffer(get("train-images-idx3-ubyte.gz"), np.uint8, offset=16).reshape(-1, 28, 28)
    labels = np.frombuffer(get("train-labels-idx1-ubyte.gz"), np.uint8, offset=8)
    for c in np.unique(labels):
        folder = Path(out) / f"class_{c}"
        folder.mkdir(parents=True, exist_ok=True)
        for i, image in enumerate(images[labels == c][:per_class]):
            Image.fromarray(image).save(folder / f"{c}_{i}.png")


# ---- Case study 1: two classifiers for the soft-voting demo ----
split_folder("data/bsm_raw", "data/bsm")
for name, weights in {"a": "yolo26n-cls.pt", "b": "yolo26s-cls.pt"}.items():
    model = YOLO(weights)
    model.train(data="data/bsm", epochs=60, imgsz=224, batch=16, patience=20, fliplr=0.5, flipud=0.5)
    shutil.copy(model.trainer.best, f"models/bsm_{name}.pt")

# ---- Case study 3: U-Net on masks made from detection boxes ----
CHIPS, SIZE = Path("data/chips"), 512
MEAN = torch.tensor([0.485, 0.456, 0.406])[:, None, None]
STD = torch.tensor([0.229, 0.224, 0.225])[:, None, None]


def ellipse_mask(img):
    parts = ["labels" if part == "images" else part for part in img.parts]
    label = Path(*parts).with_suffix(".txt")
    mask = Image.new("L", (SIZE, SIZE), 0)
    draw = ImageDraw.Draw(mask)
    if label.exists() and label.stat().st_size:
        for _, cx, cy, w, h in np.loadtxt(label, ndmin=2).reshape(-1, 5):
            draw.ellipse([(cx - w / 2) * SIZE, (cy - h / 2) * SIZE,
                          (cx + w / 2) * SIZE, (cy + h / 2) * SIZE], fill=1)
    return np.array(mask)


class ChipSet(Dataset):
    def __init__(self, paths, augment=False):
        self.paths, self.augment = paths, augment

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        image = Image.open(self.paths[i]).convert("RGB").resize((SIZE, SIZE))
        x = (torch.from_numpy(np.array(image)).permute(2, 0, 1).float() / 255 - MEAN) / STD
        y = torch.from_numpy(ellipse_mask(self.paths[i])).float()[None]
        if self.augment:                       # same change to image and mask
            k = random.randrange(4)
            x, y = x.rot90(k, (1, 2)), y.rot90(k, (1, 2))
            if random.random() < 0.5:
                x, y = x.flip(2), y.flip(2)
        return x, y


paths = sorted(p for p in (CHIPS / "images").rglob("*") if p.suffix.lower() in IMG_EXT)
random.Random(0).shuffle(paths)                # better: split by scene or area
n_val = max(1, len(paths) // 6)
train_dl = DataLoader(ChipSet(paths[n_val:], augment=True), batch_size=8, shuffle=True)
val_dl = DataLoader(ChipSet(paths[:n_val]), batch_size=8)

device = "cuda" if torch.cuda.is_available() else "cpu"
unet = smp.Unet("resnet34", encoder_weights="imagenet", in_channels=3, classes=1).to(device)
opt = torch.optim.AdamW(unet.parameters(), lr=3e-4)
dice, bce = smp.losses.DiceLoss("binary"), torch.nn.BCEWithLogitsLoss()
loss_fn = lambda logits, y: dice(logits, y) + bce(logits, y)

best = float("inf")
for epoch in range(60):
    unet.train()
    for x, y in train_dl:
        x, y = x.to(device), y.to(device)
        loss = loss_fn(unet(x), y)
        opt.zero_grad()
        loss.backward()
        opt.step()
    unet.eval()
    with torch.no_grad():
        val = sum(loss_fn(unet(x.to(device)), y.to(device)).item() for x, y in val_dl) / len(val_dl)
    print(f"epoch {epoch:2d}  validation loss {val:.3f}")
    if val < best:                             # keep the best weights (early stopping in spirit)
        best = val
        torch.save(unet.state_dict(), "models/sites_unet.pt")
```